In [ ]:
import pandas as pd
import re
import gspread
from oauth2client.service_account import ServiceAccountCredentials

def convert_google_sheet_url(url):
    pattern = r'https://docs\.google\.com/spreadsheets/d/([a-zA-Z0-9-_]+)(/edit#gid=(\d+)|/edit.*)?'
    replacement = lambda m: f'https://docs.google.com/spreadsheets/d/{m.group(1)}/export?' + (f'gid={m.group(3)}&' if m.group(3) else '') + 'format=csv'
    new_url = re.sub(pattern, replacement, url)
    return new_url

# Define function to read Google Sheets directly into a DataFrame
def read_google_sheet(url):
    # Convert the Google Sheets URL to CSV export URL
    csv_url = convert_google_sheet_url(url)

    # Load the CSV into a DataFrame
    df = pd.read_csv(csv_url)
    return df

# Replace with your Google Sheets URL
google_sheets_url = 'https://docs.google.com/spreadsheets/d'

# Read the Google Sheets document into a DataFrame
df = read_google_sheet(google_sheets_url)

# Convert the 'payment date' column to datetime if it's not already
df['payment date'] = pd.to_datetime(df['payment date'], format='%d/%m/%Y %H:%M:%S')

# Sort the DataFrame by the 'payment date' column in descending order (latest to oldest)
df_sorted = df.sort_values(by='payment date', ascending=False)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Display the sorted DataFrame
df_sorted

In [ ]:
# Grouping by 'full_name' and aggregating other columns
grouped_data = df_sorted.groupby('full_name').agg({
    'payment date': 'max',  # Assuming you want the latest payment date for each person
    'email': 'first',
    'phone': 'first',
    'college_name': 'first',
    'size_of_t_shirt': 'first',
    'item name': lambda x: list(x)
})

# Sort the DataFrame by the 'payment date' column in descending order (latest to oldest)
grouped_data = grouped_data.sort_values(by='payment date', ascending=True)

# Print the result
grouped_data

In [ ]:
# Save the grouped data to a new CSV file
grouped_data.to_csv('grouped_data.csv')